In [1]:
import pandas as pd
from pathlib import Path

project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

raw_path = (
    project_root
    / "data"
    / "raw"
    / "Dataset Capstone.xlsx"
)

processed_dir = project_root / "data" / "processed"
reports_dir = project_root / "reports"

processed_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

df_raw = pd.read_excel(
    raw_path,
    sheet_name="general_information"
)

df_clean = df_raw.copy()

print("Jumlah record awal:", len(df_clean))
print("Jumlah kolom:", len(df_clean.columns))

display(df_clean.head())

Jumlah record awal: 9
Jumlah kolom: 9


,info_id,info_key,title,content,source,source_url,accessed_at,verification_status,data_type
0,INF001,business_history,Sejarah Kopilogi,Kopilogi memulai usaha pada tahun 2015 sebagai...,wawancara,https://drive.google.com/drive/folders/1sL4ig_...,2026-11-10,pending_pic,actual
1,INF002,business_profile,Profil Kopilogi,Kopilogi merupakan usaha bidang makanan dan mi...,wawancara,https://drive.google.com/drive/folders/1sL4ig_...,2026-11-10,pending_pic,actual
2,INF003,product_types,Jenis Produk,"Kopilogi menyediakan makanan, minuman, dessert...",wawancara,https://drive.google.com/drive/folders/1sL4ig_...,2026-11-10,pending_pic,actual
3,INF004,ordering_channels,Layanan Pemesanan,Pelanggan dapat melakukan pemesanan langsung d...,wawancara,https://drive.google.com/drive/folders/1sL4ig_...,2026-11-10,pending_pic,actual
4,INF005,social_media,Media Sosial,Kopilogi menggunakan Instagram dan TikTok seba...,wawancara,https://drive.google.com/drive/folders/1sL4ig_...,2026-11-10,pending_pic,actual


In [2]:
print("=== STANDARDISASI NAMA KOLOM ===")

df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(r"\s+", "_", regex=True)
)

print(df_clean.columns.tolist())

=== STANDARDISASI NAMA KOLOM ===
['info_id', 'info_key', 'title', 'content', 'source', 'source_url', 'accessed_at', 'verification_status', 'data_type']


In [3]:
print("=== STANDARDISASI TEKS ===")

text_columns = [
    "info_id",
    "info_key",
    "title",
    "content",
    "source",
    "source_url",
    "verification_status",
    "data_type"
]

for col in text_columns:
    df_clean[col] = (
        df_clean[col]
        .astype("string")
        .str.strip()
    )

print("Standardisasi teks selesai.")

display(
    df_clean[
        ["info_id", "info_key", "title"]
    ]
)

=== STANDARDISASI TEKS ===
Standardisasi teks selesai.


,info_id,info_key,title
0,INF001,business_history,Sejarah Kopilogi
1,INF002,business_profile,Profil Kopilogi
2,INF003,product_types,Jenis Produk
3,INF004,ordering_channels,Layanan Pemesanan
4,INF005,social_media,Media Sosial
5,INF006,business_branches,Cabang Kopilogi
6,INF007,ordering_process,Proses Pemesanan
7,INF008,payment_methods,Metode Pembayaran
8,INF009,website_purpose,Tujuan Website


In [4]:
print("=== STANDARDISASI TANGGAL ===")

df_clean["accessed_at"] = pd.to_datetime(
    df_clean["accessed_at"],
    errors="coerce"
)

print(
    "Tanggal tidak valid:",
    df_clean["accessed_at"].isna().sum()
)

display(
    df_clean[
        ["info_id", "accessed_at"]
    ]
)

=== STANDARDISASI TANGGAL ===
Tanggal tidak valid: 0


,info_id,accessed_at
0,INF001,2026-11-10
1,INF002,2026-11-10
2,INF003,2026-11-10
3,INF004,2026-11-10
4,INF005,2026-11-10
5,INF006,2026-11-10
6,INF007,2026-11-10
7,INF008,2026-11-10
8,INF009,2026-11-10


In [5]:
print("=== VALIDASI GENERAL INFORMATION ===")

assert len(df_clean) == len(df_raw), \
    "Jumlah record berubah!"

assert df_clean["info_id"].is_unique, \
    "Terdapat info_id duplikat!"

assert df_clean["info_key"].is_unique, \
    "Terdapat info_key duplikat!"

assert df_clean["info_id"].notna().all(), \
    "Terdapat info_id kosong!"

assert df_clean["content"].notna().all(), \
    "Terdapat konten kosong!"

assert df_clean["content"].str.len().gt(0).all(), \
    "Terdapat konten hanya berisi spasi!"

assert df_clean["accessed_at"].notna().all(), \
    "Terdapat tanggal tidak valid!"

assert (
    df_clean["verification_status"].tolist()
    == df_raw["verification_status"].tolist()
), "Status verifikasi berubah!"

print("Jumlah record:", len(df_clean))
print("Duplikasi info_id:", df_clean["info_id"].duplicated().sum())
print("Duplikasi info_key:", df_clean["info_key"].duplicated().sum())
print("Konten kosong:", df_clean["content"].isna().sum())

print("\nValidasi struktur berhasil!")

=== VALIDASI GENERAL INFORMATION ===
Jumlah record: 9
Duplikasi info_id: 0
Duplikasi info_key: 0
Konten kosong: 0

Validasi struktur berhasil!


In [6]:
print("=== PEMERIKSAAN PERUBAHAN TEKS ===")

comparison = []

for col in text_columns:
    original = df_raw[col].astype("string")
    cleaned = df_clean[col].astype("string")

    changed = (
        original.fillna("<MISSING>")
        != cleaned.fillna("<MISSING>")
    ).sum()

    comparison.append({
        "column": col,
        "changed_values": int(changed)
    })

change_report = pd.DataFrame(comparison)

display(change_report)

=== PEMERIKSAAN PERUBAHAN TEKS ===


,column,changed_values
0,info_id,0
1,info_key,0
2,title,0
3,content,0
4,source,0
5,source_url,0
6,verification_status,0
7,data_type,0


In [7]:
print("=== MENYIMPAN DATASET BERSIH ===")

csv_path = (
    processed_dir / "general_information_clean.csv"
)

df_clean.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
    date_format="%Y-%m-%d"
)

print("Dataset berhasil disimpan:")
print(csv_path)

=== MENYIMPAN DATASET BERSIH ===
Dataset berhasil disimpan:
a:\ASAH DICODING\CAPSTONE PROJECT\kopilogi-data-science\data\processed\general_information_clean.csv


In [8]:
cleaning_log = pd.DataFrame([
    {
        "step": 1,
        "process": "Standardisasi nama kolom",
        "description": "Memastikan nama kolom konsisten",
        "result": "Struktur kolom diperiksa"
    },
    {
        "step": 2,
        "process": "Standardisasi teks",
        "description": "Menghapus spasi awal dan akhir",
        "result": "Format teks distandarkan"
    },
    {
        "step": 3,
        "process": "Standardisasi tanggal",
        "description": "Mengubah accessed_at menjadi datetime",
        "result": "Tanggal valid secara format"
    },
    {
        "step": 4,
        "process": "Validasi identitas",
        "description": "Memeriksa info_id dan info_key",
        "result": "Keunikan identitas diperiksa"
    },
    {
        "step": 5,
        "process": "Mempertahankan status verifikasi",
        "description": "Tidak mengubah pending_pic tanpa bukti",
        "result": "Metadata verifikasi dipertahankan"
    },
    {
        "step": 6,
        "process": "Pemeriksaan perubahan",
        "description": "Membandingkan nilai teks sebelum dan sesudah cleaning",
        "result": "Perubahan dicatat pada change_report"
    }
])

log_path = (
    reports_dir / "general_information_cleaning_log.csv"
)

cleaning_log.to_csv(
    log_path,
    index=False,
    encoding="utf-8-sig"
)

change_report.to_csv(
    reports_dir / "general_information_changes.csv",
    index=False,
    encoding="utf-8-sig"
)

display(cleaning_log)

print("Cleaning log berhasil disimpan.")

,step,process,description,result
0,1,Standardisasi nama kolom,Memastikan nama kolom konsisten,Struktur kolom diperiksa
1,2,Standardisasi teks,Menghapus spasi awal dan akhir,Format teks distandarkan
2,3,Standardisasi tanggal,Mengubah accessed_at menjadi datetime,Tanggal valid secara format
3,4,Validasi identitas,Memeriksa info_id dan info_key,Keunikan identitas diperiksa
4,5,Mempertahankan status verifikasi,Tidak mengubah pending_pic tanpa bukti,Metadata verifikasi dipertahankan
5,6,Pemeriksaan perubahan,Membandingkan nilai teks sebelum dan sesudah c...,Perubahan dicatat pada change_report


Cleaning log berhasil disimpan.
